In [2]:
!pip install --upgrade pennylane==0.36.0
!pip install --upgrade autoray==0.6.6
!pip install --upgrade numpy==1.26.4

     |████████████████████████████████| 1.7 MB 6.4 MB/s            
     |████████████████████████████████| 15.5 MB 167 kB/s            
  Attempting uninstall: pennylane-lightning
    Found existing installation: PennyLane-Lightning 0.38.0
    Uninstalling PennyLane-Lightning-0.38.0:
      Successfully uninstalled PennyLane-Lightning-0.38.0
  Attempting uninstall: pennylane
    Found existing installation: PennyLane 0.38.0
    Uninstalling PennyLane-0.38.0:
      Successfully uninstalled PennyLane-0.38.0
You should consider upgrading via the '/faststorage/project/DEIC-SDU-L2-22/env/bin/python3 -m pip install --upgrade pip' command.
     |████████████████████████████████| 54 kB 2.9 MB/s             
  Attempting uninstall: autoray
    Found existing installation: autoray 0.8.1
    Uninstalling autoray-0.8.1:
      Successfully uninstalled autoray-0.8.1
You should consider upgrading via the '/faststorage/project/DEIC-SDU-L2-22/env/bin/python3 -m pip install --upgrade pip' command.
You s

In [1]:
# hybrid_llm_quantum_grover.py
import re
import math
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pennylane as qml
from pennylane import numpy as pnp

# ----------------------------
# Config et utilitaires
# ----------------------------
# Plage des ratings dans ton dataset (ex: 0..5)
MIN_RATING = 0
MAX_RATING = 5
NUM_RATINGS = MAX_RATING - MIN_RATING + 1  # ici 6

# Nombre de qubits nécessaires pour représenter NUM_RATINGS états
n_qubits = math.ceil(math.log2(NUM_RATINGS))
print(f"Num ratings = {NUM_RATINGS}, n_qubits = {n_qubits} (2^n = {2**n_qubits})")

# Device PennyLane (simulateur)
dev = qml.device("default.qubit", wires=n_qubits, shots=1024)


/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/faststorage/project/DEIC-SDU-L2-22/env/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


Num ratings = 6, n_qubits = 3 (2^n = 8)


In [4]:
# Util : convertit un rating entier en bitlist de longueur n_qubits
def rating_to_bits(rating, n_qubits):
    v = int(round(rating - MIN_RATING))  # index 0..NUM_RATINGS-1
    b = format(v, f"0{n_qubits}b")
    return [int(ch) for ch in b]

# Util : convertit bitlist en rating entier
def bits_to_rating(bits):
    idx = int("".join(str(b) for b in bits), 2)
    return MIN_RATING + idx

In [5]:
# ----------------------------
# LLM : generation + extraction
# ----------------------------
# NOTE: pour démo on charge un modèle léger. Remplace par ton Mistral si dispo.
LLM_NAME = "distilgpt2"  # pour test local rapide ; remplace par "filipealmeida/Mistral-7B-Instruct-v0.1-sharded" si tu peux
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(LLM_NAME)

def generate_prompt(user_row, movie_row):
    # exemple simple de prompt ; adapte selon ton format Alpaca si tu veux
    prompt = (
        f"User: sex={user_row.get('sex','Unknown')}, age={user_row.get('age','?')}, country={user_row.get('Country','?')}, mood={user_row.get('mood','?')}\n"
        f"Movie: title={movie_row.get('Movie_Name','?')}, director={movie_row.get('Director','?')}, genres={movie_row.get('Genre1','?')}/{movie_row.get('Genre2','?')}\n"
        "Question: What rating (0-5) would this user give this movie? Answer with a single number between 0 and 5.\n"
    )
    return prompt


In [6]:
def extract_rating_from_text(text):
    # cherche un nombre 0-5 (float possible). On prend la première occurrence raisonnable.
    # Robustifier si le modèle donne du texte plus complexe.
    m = re.search(r"(?<!\d)([0-5](?:\.\d+)?)", text)
    if m:
        return float(m.group(1))
    # fallback : si échec, retourne la moyenne
    return float((MIN_RATING + MAX_RATING) / 2)

In [7]:
def llm_predict_rating(user_row, movie_row, max_new_tokens=16):
    prompt = generate_prompt(user_row, movie_row)
    inputs = tokenizer(prompt, return_tensors="pt")
    # génération simple (attention: certains modèles ont tokens spéciaux)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # On peut vouloir ne garder que la partie après "Answer:"
    if "Answer" in decoded:
        decoded = decoded.split("Answer", 1)[1]
    rating = extract_rating_from_text(decoded)
    # clamp
    rating = max(MIN_RATING, min(MAX_RATING, rating))
    return rating, decoded


In [8]:
# ----------------------------
# Opérateurs quantiques : Oracle & Diffusion
# ----------------------------
def apply_oracle(target_bits):
    """
    Retourne une fonction qui applique l'oracle marquant target_bits
    (phase flip on |target_bits>).
    """
    def _oracle():
        # Pour marquer l'état target, on applique X sur qubits où bit==0,
        # ensuite on applique MultiControlledZ (réalisée via MultiControlledX + H),
        # puis on remet les X inverses.
        # qml.MultiControlledX est disponible dans PennyLane >= certain version.
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
        # multi-controlled Z via H on last wire + multi-controlled X:
        if n_qubits == 1:
            qml.PauliZ(wires=0)
        else:
            # transforme Z_controlled en X controlled grâce à H sur target (last wire)
            target = n_qubits - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, n_qubits - 1))
            # si n_qubits-1 == 0, on applique un X contrôlé simple ; PennyLane gère MultiControlledX
            qml.MultiControlledX(wires=controls + [target])
            qml.Hadamard(wires=target)
        for i, b in enumerate(target_bits):
            if b == 0:
                qml.PauliX(wires=i)
    return _oracle

In [9]:
def diffusion():
    """
    Opérateur de diffusion de Grover (inversion about average).
    """
    def _diffuse():
        # H^{\otimes n}
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
        # X^{\otimes n}
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        # multi-controlled Z
        if n_qubits == 1:
            qml.PauliZ(wires=0)
        else:
            target = n_qubits - 1
            qml.Hadamard(wires=target)
            controls = list(range(0, n_qubits - 1))
            qml.MultiControlledX(wires=controls + [target])
            qml.Hadamard(wires=target)
        # X^{\otimes n}
        for i in range(n_qubits):
            qml.PauliX(wires=i)
        # H^{\otimes n}
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
    return _diffuse

In [10]:
# QNode qui applique Grover avec k itérations pour une target donnée
def make_grover_qnode(k_iterations):
    @qml.qnode(dev, interface="autograd")
    def grover_circuit(target_bits):
        # Initial superposition on the first 2^n states
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
        # Grover iterations
        oracle_fn = apply_oracle(target_bits)
        diffuse_fn = diffusion()
        for _ in range(k_iterations):
            oracle_fn()
            diffuse_fn()
        # retourne fréquences/probas par mesure en computational basis (par shots)
        return [qml.sample(qml.PauliZ(i)) for i in range(n_qubits)]
    return grover_circuit


In [11]:
# Note: on récupère ici les samples et on transforme en distribution empirique
def run_grover_and_get_predicted_rating(target_bits, k_iter=1):
    # On choisit k_iter ~ floor(pi/4 * sqrt(N/M)) mais ici M=1, N=2^n
    qnode = make_grover_qnode(k_iter)
    # On exécute avec shots défini dans device ; qnode retourne des samples par qubit
    samples = qnode(target_bits)  # liste de arrays (one per wire)
    # samples shape: list of n_qubits arrays of length shots, each value in {-1,1}
    # transform to bits (0/1): (1 -> 0, -1 -> 1) depending on measurement basis PauliZ gives +1 for |0>
    shots = len(samples[0])
    bitstrings = []
    for s in range(shots):
        bits = []
        for q in range(n_qubits):
            val = samples[q][s]
            # val is +1 for |0>, -1 for |1>
            bit = 0 if val > 0 else 1
            bits.append(bit)
        bitstrings.append("".join(str(b) for b in bits))
    # comptage des bitstrings
    from collections import Counter
    counts = Counter(bitstrings)
    most_common_bits = counts.most_common(1)[0][0]
    predicted_bits = [int(ch) for ch in most_common_bits]
    predicted_rating = bits_to_rating(predicted_bits)
    return predicted_rating, counts

In [12]:
# inspiré du code nousaiba
# ----------------------------
# Pipeline complet : appliquer au dataset
# ----------------------------
def hybrid_predict_for_row(user_row, movie_row, grover_iters=1, weight_llm=0.5):
    # 1) LLM prédiction (float)
    llm_rating, llm_text = llm_predict_rating(user_row, movie_row)
    
    # 2) Quantifier / target pour l'oracle (arrondi)
    target_int = int(round(llm_rating))
    target_int = max(MIN_RATING, min(MAX_RATING, target_int))
    target_bits = rating_to_bits(target_int, n_qubits)
    
    # 3) Lancer Grover (oracle ciblé) -> prédiction quantique discrete
    q_pred, counts = run_grover_and_get_predicted_rating(target_bits, k_iter=grover_iters)
    
    # 4) Combinaison : exemple simple moyenne pondérée
    final_pred = weight_llm * llm_rating + (1.0 - weight_llm) * float(q_pred)
    
    return {
        "llm_rating": llm_rating,
        "llm_text": llm_text,
        "target_int": target_int,
        "q_pred": q_pred,
        "counts": counts,
        "final_pred": final_pred
    }


In [13]:
# ----------------------------
# Exemple d'exécution sur un petit DataFrame
# ----------------------------
if __name__ == "__main__":
    # Exemple minimal : crée un df avec 2 lignes
    df = pd.DataFrame([
        {"userID":26, "sex":"Female", "age":26, "Country":"United Kingdom", "mood":"Neutral",
         "Movie_Name":"Linus Roache Movie", "Director":"Antonia Bird", "Genre1":"Drama"},
        {"userID":26, "sex":"Female", "age":26, "Country":"United States", "mood":"Neutral",
         "Movie_Name":"Geek Charming", "Director":"K. Asher Levin", "Genre1":"Comedy"}
    ])
    
    results = []
    for idx, row in df.iterrows():
        # Ici on passe la même row comme user_row / movie_row pour la demo
        res = hybrid_predict_for_row(row, row, grover_iters=1, weight_llm=0.6)
        print(f"Row {idx} -> LLM:{res['llm_rating']:.3f}, Q_pred:{res['q_pred']}, final:{res['final_pred']:.3f}")
        results.append(res)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Row 0 -> LLM:0.000, Q_pred:0, final:0.000
Row 1 -> LLM:0.000, Q_pred:0, final:0.000
